# Chapter 12 — Reductions I: Warp-Level Primitives

> Course: **llm.c — Zero to Hero**, Chapter 12 of ~20.
> Builds on Chapters 9–11 (kernels, grid-stride, coalescing).

GELU and the encoder are *embarrassingly parallel*: thread `i` only touches `inp[i]` and writes `out[i]`. Softmax and LayerNorm are different — they need **per-row sums and maxes**, computed across many elements. That's a **reduction**.

Doing a reduction efficiently on a GPU means understanding **warps** (32 threads in lockstep) and the **shuffle instructions** that let warps exchange register values directly, without going through shared memory or global memory. We can sum 32 values across a warp in `log2(32) = 5` instructions. This is one of the prettiest patterns in CUDA.

This chapter teaches just the warp-level mechanics. Chapter 13 builds on it for full block-level reductions.

### Learning objectives

By the end of this chapter you will:

- Explain what a warp is and why threads in a warp execute in lockstep.
- Use `__shfl_down_sync` to sum 32 values across a warp in 5 instructions.
- Implement `warpReduceSum` and `warpReduceMax` from scratch.
- Read and understand `softmax_forward_kernel5` from `dev/cuda/softmax_forward.cu`.


## 1. Concept — Warps and Lockstep Execution

A **warp** is the smallest scheduling unit on an NVIDIA GPU: **32 threads** that always execute the *same instruction* at the same clock cycle. They live in the same register file, share an instruction pointer, and cannot diverge in the ALU pipeline (though they can take different code paths via predication, with a perf cost).

This lockstep guarantee enables a magical class of instructions: **warp shuffles** (`__shfl_*`). They let one thread *read another thread's register* directly, with no memory traffic.

The most useful one for reductions is `__shfl_down_sync(mask, value, offset)`:

> "Each thread asks the thread that's `offset` higher in the warp for its `value`. Returns that value."

So if thread 0 has `value = 5.0` and thread 16 has `value = 7.0`, then `__shfl_down_sync(0xffffffff, value, 16)` returns `7.0` to thread 0 (and undefined to thread 16, which has nobody at offset 32).

The `mask` argument is a **lane mask** of which threads participate — almost always `0xffffffff` (all 32 lanes).


## 2. Warp Reduction in 5 Instructions

To sum 32 values in a warp:

```c
__device__ float warpReduceSum(float val) {
    val += __shfl_down_sync(0xffffffff, val, 16);   // each thread t adds thread t+16's value
    val += __shfl_down_sync(0xffffffff, val,  8);   // ... thread t+8's value
    val += __shfl_down_sync(0xffffffff, val,  4);
    val += __shfl_down_sync(0xffffffff, val,  2);
    val += __shfl_down_sync(0xffffffff, val,  1);
    return val;        // thread 0 holds the full sum across the warp
}
```

Trace it for an 8-thread "warp" (real warps are 32, but the pattern is the same):

```
initial:    [a0, a1, a2, a3, a4, a5, a6, a7]
+offset 4:  [a0+a4, a1+a5, a2+a6, a3+a7, _, _, _, _]      threads 0..3 now hold pairs
+offset 2:  [a0+a4+a2+a6, a1+a5+a3+a7, _, _, ...]         threads 0..1 hold halves
+offset 1:  [a0+...+a7,   _, _, _, ...]                    thread 0 holds full sum
```

This is a **butterfly reduction**. After `log2(32) = 5` shuffles, lane 0 holds `Σ`. The other lanes hold partial sums you usually ignore.

Same pattern for max:

```c
__device__ float warpReduceMax(float val) {
    val = fmaxf(val, __shfl_down_sync(0xffffffff, val, 16));
    val = fmaxf(val, __shfl_down_sync(0xffffffff, val,  8));
    val = fmaxf(val, __shfl_down_sync(0xffffffff, val,  4));
    val = fmaxf(val, __shfl_down_sync(0xffffffff, val,  2));
    val = fmaxf(val, __shfl_down_sync(0xffffffff, val,  1));
    return val;
}
```

For any associative+commutative operation: addition, max, min, multiplication, bitwise-or, etc.

These are the building blocks of every meaningful CUDA kernel that does any kind of reduction. `llm.c` uses them everywhere — every softmax, every LayerNorm, every gradient norm.


## 3. Demo — Warp Reduction in Action

In [ ]:
!mkdir -p course/ch12_build


In [ ]:
%%writefile course/ch12_build/warp_reduce.cu
#include <stdio.h>
#include <cuda_runtime.h>

__device__ float warpReduceSum(float val) {
    for (int offset = 16; offset > 0; offset /= 2)
        val += __shfl_down_sync(0xffffffff, val, offset);
    return val;
}

__device__ float warpReduceMax(float val) {
    for (int offset = 16; offset > 0; offset /= 2)
        val = fmaxf(val, __shfl_down_sync(0xffffffff, val, offset));
    return val;
}

// Each block has exactly 1 warp (32 threads). It loads 32 values,
// reduces them, and lane 0 writes the result.
__global__ void warp_sum_kernel(float* out, const float* inp) {
    int t = threadIdx.x;          // lane id 0..31
    float val = inp[t];
    val = warpReduceSum(val);
    if (t == 0) out[blockIdx.x] = val;
}

__global__ void warp_max_kernel(float* out, const float* inp) {
    int t = threadIdx.x;
    float val = inp[t];
    val = warpReduceMax(val);
    if (t == 0) out[blockIdx.x] = val;
}

int main(void) {
    // 32-element input
    float h_inp[32];
    for (int i = 0; i < 32; i++) h_inp[i] = (float)(i + 1);  // 1..32, sum = 32*33/2 = 528, max = 32

    float *d_inp, *d_sum, *d_max;
    cudaMalloc(&d_inp, 32*4); cudaMalloc(&d_sum, 4); cudaMalloc(&d_max, 4);
    cudaMemcpy(d_inp, h_inp, 32*4, cudaMemcpyHostToDevice);

    warp_sum_kernel<<<1, 32>>>(d_sum, d_inp);
    warp_max_kernel<<<1, 32>>>(d_max, d_inp);

    float h_sum, h_max;
    cudaMemcpy(&h_sum, d_sum, 4, cudaMemcpyDeviceToHost);
    cudaMemcpy(&h_max, d_max, 4, cudaMemcpyDeviceToHost);
    printf("warp sum: %.1f (expected 528)\n", h_sum);
    printf("warp max: %.1f (expected 32)\n",  h_max);

    cudaFree(d_inp); cudaFree(d_sum); cudaFree(d_max);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch12_build/warp_reduce course/ch12_build/warp_reduce.cu && ./course/ch12_build/warp_reduce


You should see `528` and `32`. The whole reduction took **5 instructions** per thread (the 5 shuffles). No shared memory, no `__syncthreads()`, no global atomics. Just register-to-register communication within the warp.


## 4. Demo — Softmax Using Warp Reductions

Now let's actually use this for something real. Softmax over 32 values fits perfectly in a warp:

1. Each thread loads one element of the row.
2. `warpReduceMax` to find the row max (for stability).
3. Each thread computes `exp(val - max)`.
4. `warpReduceSum` to find the sum of exp values.
5. Each thread divides by the sum and writes its output.

This is a single-warp version of `softmax_forward_kernel5` in `dev/cuda/softmax_forward.cu`. The real one handles bigger rows, but the core idea is identical.


In [ ]:
%%writefile course/ch12_build/warp_softmax.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

__device__ float warpReduceSum(float val) {
    for (int offset = 16; offset > 0; offset /= 2) val += __shfl_down_sync(0xffffffff, val, offset);
    return val;
}
__device__ float warpReduceMax(float val) {
    for (int offset = 16; offset > 0; offset /= 2) val = fmaxf(val, __shfl_down_sync(0xffffffff, val, offset));
    return val;
}

// 1 warp = 1 row. Row size must be 32. logits is (B*T, 32), probs is (B*T, 32).
__global__ void softmax_warp(float* probs, const float* logits, int n_rows) {
    int row = blockIdx.x;     // one block per row
    int t   = threadIdx.x;    // 0..31, one thread per element

    float v = logits[row * 32 + t];

    // pass 1: row max
    float m = warpReduceMax(v);
    // shuffle to broadcast lane 0's m to all lanes
    m = __shfl_sync(0xffffffff, m, 0);

    float ev = expf(v - m);

    // pass 2: row sum
    float s = warpReduceSum(ev);
    s = __shfl_sync(0xffffffff, s, 0);

    probs[row * 32 + t] = ev / s;
}

int main(void) {
    int B = 4;     // 4 rows, each of length 32
    int N = B * 32;
    float* h_logits = (float*) malloc(N*4);
    float* h_gpu    = (float*) malloc(N*4);
    float* h_cpu    = (float*) malloc(N*4);
    for (int i = 0; i < N; i++) h_logits[i] = (float)((i*7) % 13) - 6.0f;

    // CPU reference
    for (int r = 0; r < B; r++) {
        float m = -1e30f;
        for (int i = 0; i < 32; i++) m = fmaxf(m, h_logits[r*32+i]);
        float s = 0.0f;
        for (int i = 0; i < 32; i++) { h_cpu[r*32+i] = expf(h_logits[r*32+i] - m); s += h_cpu[r*32+i]; }
        for (int i = 0; i < 32; i++) h_cpu[r*32+i] /= s;
    }

    float *d_logits, *d_probs;
    cudaMalloc(&d_logits, N*4); cudaMalloc(&d_probs, N*4);
    cudaMemcpy(d_logits, h_logits, N*4, cudaMemcpyHostToDevice);

    softmax_warp<<<B, 32>>>(d_probs, d_logits, B);

    cudaMemcpy(h_gpu, d_probs, N*4, cudaMemcpyDeviceToHost);
    float maxerr = 0;
    for (int i = 0; i < N; i++) {
        float e = fabsf(h_gpu[i] - h_cpu[i]);
        if (e > maxerr) maxerr = e;
    }
    printf("softmax max diff: %.2e\n", maxerr);
    // also check rows sum to 1
    for (int r = 0; r < B; r++) {
        float s = 0; for (int i = 0; i < 32; i++) s += h_gpu[r*32+i];
        printf("row %d sum: %.6f\n", r, s);
    }

    cudaFree(d_logits); cudaFree(d_probs);
    free(h_logits); free(h_gpu); free(h_cpu);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch12_build/warp_softmax course/ch12_build/warp_softmax.cu && ./course/ch12_build/warp_softmax


All four rows should sum to exactly `1.0`, and the diff vs the CPU reference should be near float32 noise. **Softmax in 7 lines of GPU code.**

Notice the `__shfl_sync(0xffffffff, m, 0)` call — that's a different warp shuffle that **broadcasts** lane 0's value to all lanes. After `warpReduceSum`, only lane 0 holds the result; we use shuffle-broadcast to share it back so every thread can divide.


## 5. Translation Bridge

| Operation | CPU code | GPU warp code |
|---|---|---|
| Sum N values | `for (i) sum += a[i];` | 1 thread per element + 5 `__shfl_down_sync` ops |
| Max of N values | `for (i) m = max(m, a[i]);` | Same idea, with `fmaxf` |
| Per-row softmax | 3-pass scalar loop | 7-line single-warp kernel |
| Communication | shared memory or message-passing | `__shfl_*_sync` register-level |

The big idea: **for reductions of up to 32 elements, warps + shuffles are essentially free.** No memory traffic, no synchronization, just register-to-register data movement in the existing 32-wide vector pipe.


## 6. TODO Exercise — Warp Mean

In [ ]:
%%writefile course/ch12_build/exercise1.cu
#include <stdio.h>
#include <cuda_runtime.h>

// TODO: implement warp_mean by composing warpReduceSum and a divide.
__device__ float warpReduceSum(float val) {
    // TODO: 5 shuffle steps with offsets 16, 8, 4, 2, 1
    return val;
}

__device__ float warp_mean(float val) {
    // TODO: warpReduceSum then divide by 32
    return val;
}

__global__ void test_kernel(float* out, const float* inp) {
    int t = threadIdx.x;
    float v = inp[t];
    v = warp_mean(v);
    if (t == 0) out[0] = v;
}

int main(void) {
    float h_inp[32];
    for (int i = 0; i < 32; i++) h_inp[i] = 2.0f * (i + 1);   // 2..64, mean = 33
    float *d_inp, *d_out;
    cudaMalloc(&d_inp, 32*4); cudaMalloc(&d_out, 4);
    cudaMemcpy(d_inp, h_inp, 32*4, cudaMemcpyHostToDevice);
    test_kernel<<<1, 32>>>(d_out, d_inp);
    float h_out;
    cudaMemcpy(&h_out, d_out, 4, cudaMemcpyDeviceToHost);
    printf("warp mean: %.3f (expected 33.000)\n", h_out);
    cudaFree(d_inp); cudaFree(d_out);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch12_build/exercise1 course/ch12_build/exercise1.cu && ./course/ch12_build/exercise1


### Solution

In [ ]:
%%writefile course/ch12_build/exercise1_sol.cu
#include <stdio.h>
#include <cuda_runtime.h>

__device__ float warpReduceSum(float val) {
    for (int offset = 16; offset > 0; offset /= 2)
        val += __shfl_down_sync(0xffffffff, val, offset);
    return val;
}

__device__ float warp_mean(float val) {
    return warpReduceSum(val) * (1.0f / 32.0f);
}

__global__ void test_kernel(float* out, const float* inp) {
    int t = threadIdx.x;
    float v = inp[t];
    v = warp_mean(v);
    if (t == 0) out[0] = v;
}

int main(void) {
    float h_inp[32];
    for (int i = 0; i < 32; i++) h_inp[i] = 2.0f * (i + 1);
    float *d_inp, *d_out;
    cudaMalloc(&d_inp, 32*4); cudaMalloc(&d_out, 4);
    cudaMemcpy(d_inp, h_inp, 32*4, cudaMemcpyHostToDevice);
    test_kernel<<<1, 32>>>(d_out, d_inp);
    float h_out;
    cudaMemcpy(&h_out, d_out, 4, cudaMemcpyDeviceToHost);
    printf("warp mean: %.3f (expected 33.000)\n", h_out);
    cudaFree(d_inp); cudaFree(d_out);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch12_build/exercise1_sol course/ch12_build/exercise1_sol.cu && ./course/ch12_build/exercise1_sol


## Recap

You now know:

- **Warp = 32 threads in lockstep**, with shared registers and special "shuffle" instructions.
- **`__shfl_down_sync`** lets thread `t` read thread `t+offset`'s register value, with zero memory traffic.
- A **butterfly reduction** sums 32 values across a warp in `log2(32) = 5` shuffles. Same pattern works for max, min, multiply, OR.
- Softmax over a 32-wide row fits in a single 7-line CUDA kernel.

### What's next

**Chapter 13 — Reductions II: Shared Memory & Block Reductions.** What if your row is bigger than 32? Use a **block reduction**: each warp does a warp-level reduction, then a single warp reduces those partial results, all coordinated through **shared memory**. We'll write `blockReduceSum` and use it to build the per-block LayerNorm reduction that `llm.c`'s `layernorm_forward_kernel6` uses.

When you're ready, say **"proceed to Chapter 13"**.
